In [5]:
# LOADS & IMPORTS
%load_ext autoreload
%autoreload 2

import params
import data
import constraints
import model
import supply_risk
import plots

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# SOLVER
m = model.build_model(params)

from pyomo.environ import SolverFactory, value
solver = SolverFactory('glpk')
result = solver.solve(m)

print(result.solver.status)
print("The total minimum cost is :", value(m.obj_cost), "€.")

ok
The total minimum cost is : 3200.0 €.


In [7]:
# SOLUTION CHECK

# Which enrichment plant was built?
for e in m.E:
    for t in m.T:
        v = value(m.Y_et[e,t])
        print(f"Y_et[{e},{t}] = {v}")
        if v == 1.0:
            print(f"  → facility {e},{t} was built")

# What are the flows between L and ET?
for l in m.L:
    for e in m.E:
        for t in m.T:
            q = value(m.Q_let[l,e,t])
            print(f"Q_let[{l},{e},{t}] = {q}")
            if q > 0:
                print(f"  → {q} kg shipped from {l} to {e},{t}")

# What are the flows between ET and R?
for e in m.E:
    for t in m.T:
        for r in m.R:
            q = value(m.Q_etr[e,t,r])
            print(f"Q_etr[{e},{t},{r}] = {q}")
            if q > 0:
                print(f"  → {q} kg shipped from {e},{t} to {r}")


# Does does each site e's sourcing mix add up to 100%?
for e in m.E:
    f_e = supply_risk.import_share_L(m, e)
    print(e, f_e, "→ sum =", sum(f_e.values()))

Y_et[e1,t1] = 1.0
  → facility e1,t1 was built
Y_et[e1,t2] = 0.0
Y_et[e2,t1] = 1.0
  → facility e2,t1 was built
Y_et[e2,t2] = 0.0
Q_let[l1,e1,t1] = 10.0
  → 10.0 kg shipped from l1 to e1,t1
Q_let[l1,e1,t2] = 0.0
Q_let[l1,e2,t1] = 10.0
  → 10.0 kg shipped from l1 to e2,t1
Q_let[l1,e2,t2] = 0.0
Q_let[l2,e1,t1] = 0.0
Q_let[l2,e1,t2] = 0.0
Q_let[l2,e2,t1] = 0.0
Q_let[l2,e2,t2] = 0.0
Q_etr[e1,t1,r1] = 5.0
  → 5.0 kg shipped from e1,t1 to r1
Q_etr[e1,t2,r1] = 0.0
Q_etr[e2,t1,r1] = 5.0
  → 5.0 kg shipped from e2,t1 to r1
Q_etr[e2,t2,r1] = 0.0
e1 {'l1': 1.0, 'l2': 0.0} → sum = 1.0
e2 {'l1': 1.0, 'l2': 0.0} → sum = 1.0


In [8]:
# SR-ONLY SOLVE
from pyomo.environ import Objective, minimize

# "Which soure does the model choose when only supply risk matters —
#  and what does that solution cost compared to the cost-optimal one?"

m2 = model.build_model(params)
m2.obj_cost.deactivate()                  # switch off cost objective
m2.obj_SR.activate()                      # switch on supply risk objective
solver.solve(m2)

print("min-cost run: cost =", value(m.obj_cost),  "€ | SR =", value(m.obj_SR))
print("min-SR run:   cost =", value(m2.obj_cost), "€ | SR =", value(m2.obj_SR))

min-cost run: cost = 3200.0 € | SR = 0.1
min-SR run:   cost = 3400.0 € | SR = 0.025


In [9]:
# COST-ONLY SOLVE (LEXICOGRAPHIC-PAYOFF-TABLE)

from pyomo.environ import Objective, minimize, Constraint

# "Which source does the model choose when only costs matter —
#  and how risky is that solution compared to the risk-optimal one?"

m3 = model.build_model(params)
m3.obj_cost.activate()                          # defaults: obj_cost on, obj_SR off
m3.obj_SR.deactivate()
solver.solve(m3)                                # round 1
cost_star = value(m3.obj_cost)

m3.c_lex = Constraint(expr = m3.obj_cost.expr == cost_star)  # freeze cost at its optimum
m3.obj_cost.deactivate()
m3.obj_SR.activate()
solver.solve(m3)                                # round 2

print(f"Cheapest possible supply costs {cost_star} €.")
print(f"Among ALL solutions that cost exactly {cost_star} €, the safest has SR = {value(m3.obj_SR)}.")
print(f"→ ε-range for AUGMECON: [{value(m2.obj_SR)}, {value(m3.obj_SR)}]")

Cheapest possible supply costs 3200.0 €.
Among ALL solutions that cost exactly 3200.0 €, the safest has SR = 0.09999999999999987.
→ ε-range for AUGMECON: [0.025, 0.09999999999999987]
